In [1]:
import cv2
import yolov5
import easyocr
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
# load model
model = yolov5.load('./model.pt') 

YOLOv5  2023-6-2 Python-3.9.13 torch-2.0.1+cpu CPU

c:\Users\mkumb\Documents\Projects\python\nikumbusheDeni\env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fusing layers... 
Model summary: 212 layers, 20852934 parameters, 0 gradients, 47.9 GFLOPs
Adding AutoShape... 


In [3]:
reader = easyocr.Reader(["en"])

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


In [4]:
# set model parameters
model.conf = 0.25  # NMS confidence threshold
model.iou = 0.45  # NMS IoU threshold
model.agnostic = False  # NMS class-agnostic
model.multi_label = False  # NMS multiple labels per box
model.max_det = 1000  # maximum number of detections per image

In [5]:
# set image path
img = './test_videos/test_pic3.jpg'

In [6]:
def load_img(path):
    the_img = cv2.imread(path)
    # cv2.resize(the_img, (120, 120))
    return the_img

In [7]:
def plate_detection(path: str):
    # Read image
    image = load_img(path)
    
    # perform inference
    # results = model(img)

    # inference with larger input size
    # results = model(img, size=640)

    # inference with test time augmentation
    results = model(image, augment=True)

    # parse results
    predictions = results.pred[0]
    boxes = predictions[:, :4] # x1, y1, x2, y2
    scores = predictions[:, 4]
    categories = predictions[:, 5]
    
    # getting bounding box coordinates
    xmin, ymin, xmax, ymax = map(int, boxes.tolist()[0])
    pt1 =(xmin,ymin)
    pt2 =(xmax,ymax)

    # cropped_image = image[ymin:ymax, xmin:xmax]
    cropped_image = cv2.cvtColor(image[ymin:ymax, xmin:xmax], cv2.COLOR_BGR2RGB)
    
    cv2.rectangle(image, pt1, pt2, (0, 0, 255), 10)
    cv2.putText(image, "Licence", pt1, cv2.FONT_HERSHEY_SIMPLEX, 3, (0, 0, 255), 10)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return cropped_image, image, boxes

In [8]:
cropped, image, cods = plate_detection(img)
plt.imshow(cropped)

### Commented out this cell
`Its Purpose was to obtain a good grayscaling for an easy way for OCR to extract text.`

`Found None out of them`

```python
gray_cropped_image = cv2.cvtColor(cropped, cv2.COLOR_RGB2GRAY)
ret,thresh1 = cv2.threshold(gray_cropped_image,127,255,cv2.THRESH_BINARY)
ret,thresh2 = cv2.threshold(gray_cropped_image,127,255,cv2.THRESH_BINARY_INV)
ret,thresh3 = cv2.threshold(gray_cropped_image,127,255,cv2.THRESH_TRUNC)
ret,thresh4 = cv2.threshold(gray_cropped_image,127,255,cv2.THRESH_TOZERO)
ret,thresh5 = cv2.threshold(gray_cropped_image,127,255,cv2.THRESH_TOZERO_INV)

titles = ['Original Image','BINARY with OTSU','BINARY_INV','TRUNC from OTSU','TOZERO','TOZERO_INV']
images = [cropped, thresh1, thresh2, thresh3, thresh4, thresh5]

for i in range(6):
    plt.subplot(2,3,i+1),plt.imshow(images[i],'gray',vmin=0,vmax=255)
    plt.title(titles[i])
    plt.xticks([]),plt.yticks([])
```

In [12]:
results = reader.readtext(cropped)

In [15]:
for (bbox, text, prob) in results:
    # Extract coordinates
    bbox = np.round(bbox).astype(np.int32)
    # print(bbox, text)
    top_left = tuple(bbox[0])
    bottom_right = tuple(bbox[2])
    # print(top_left, bottom_right)

    # Draw bounding box and text
    cv2.rectangle(cropped, top_left, bottom_right, (0, 255, 0), 2)
    # cv2.putText(cropped_image, text, top_left, cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

results

[([[161.92307692307693, 34.61538461538461],
   [254.97913833770846, 6.738356332881776],
   [272.0769230769231, 73.38461538461539],
   [179.02086166229154, 102.26164366711822]],
  'ALM',
  0.8538817526858696),
 ([[28.94383898749285, 75.10452627873785],
   [166.00352237382504, 37.712494056080686],
   [182.05616101250715, 105.89547372126215],
   [44.99647762617495, 144.2875059439193]],
  'T517',
  0.4905734062194824)]

In [ ]:
plt.imshow(cropped)

# Key Take Aways
- The algorithm, can Detect the plate number in multiple orientations
- Detecting plates on multiple orientations means It can even start to extract plate letters then numbers

In [14]:
# save results into "results/" folder
# results.save(save_dir='results/')